# Analyzing and Predicting Annual Corn Prices

This notebook loads USDA NASS annual corn price data (1866-2025), adjusts prices for inflation using the Consumer Price Index (CPI), and applies two regression models for time series analyses, OLS linear time trend and an AR(1) autoregressive model, to explore and forecast real corn prices.

Click the badge below to open in Google Colab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chuckgrigsby0/agec-370/blob/main/notebooks/19_predictive_modeling_corn_prices.ipynb)

## Data Loading

Import libraries and load the corn price and CPI datasets from the project's GitHub repository.

In [ ]:
# Import data manipulation libraries
import pandas as pd
import numpy as np

# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Load data from GitHub repository
base_url = "https://raw.githubusercontent.com/chuckgrigsby0/agec-370/main/data/"
corn_prices = pd.read_csv(base_url + 'corn_prices_bushels.csv')

# Load CPI data directly from GitHub URL
cpi = pd.read_csv(base_url + 'cpi_1913_2024.csv')

In [ ]:
# Alternatively, if you have the data in Google Drive, you can load it using the following code:
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('drive/MyDrive/Data/my_data.csv')

## Data Cleaning

The `Value` column from NASS data often contains suppressed entries marked `(D)` or `(Z)`, comma-formatted numbers, and mixed types. The steps below standardize it into a clean numeric column.

In [ ]:
value_clean = corn_prices['Value'].fillna('').astype(str)

# Remove any row containing (D) or (Z) anywhere
mask = value_clean.str.contains(r'\([DZ]\)', regex=True)
corn_prices = corn_prices[~mask].copy()

# Remove ',' from `Value` column
corn_prices['Value'] = (
    corn_prices['Value']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
# Convert to numeric
corn_prices['Value'] = pd.to_numeric(corn_prices['Value'], errors='coerce')

# Check for NAs
print(corn_prices['Value'].isna().sum())

# Optional: Drop NAs if needed
# corn_prices.dropna(subset=['Value'], inplace=True) 

### Rename `Value` to `nominal_price`

Let's rename the `Value` column to `nominal_price` before we add the CPI-adjusted `real_price` column later.

In [ ]:
corn_prices.rename(columns={'Value': 'nominal_price'}, inplace=True)

## Exploring the Dataset

Before merging additional the CPI data, let's look at the structure of our DataFrame using several pandas and NumPy methods.


In [ ]:
# .shape returns (rows, columns) as a tuple
print("Shape:", corn_prices.shape)

# .dtypes shows each column's data type: int64, float64, or object (strings/mixed)
print("\nData types:\n", corn_prices.dtypes)


In [ ]:
# .head() shows the first 5 rows (default); use .head(n) for a custom number
corn_prices.head()


In [ ]:
# .tail() shows the last 5 rows
corn_prices.tail()


In [ ]:
# .info() shows column names, non-null counts, and dtype for every column at once
# Use this early to catch columns with unexpected missing values
corn_prices.info()


In [ ]:
# Select a single column → returns a pandas Series (one-dimensional)
# corn_prices['nominal_price']

# Select multiple columns → returns a DataFrame (two-dimensional)
corn_prices[['year', 'nominal_price']].head()


In [ ]:
# Boolean row filter: create a True/False mask, then apply it to the DataFrame
# Keep only years where nominal price exceeded $5.00/bu
high_price_years = corn_prices[corn_prices['nominal_price'] > 5.0]
high_price_years[['year', 'nominal_price']]


In [ ]:
# .unique() returns an array of all distinct values in a Series
# corn_prices['year'].unique()

# .min() and .max() give the minimum and maximum values in a Series
print("Year range:", corn_prices['year'].min(), "–", corn_prices['year'].max())
# .nunique() counts how many unique values there are
print("Unique years:", corn_prices['year'].nunique())

# Check the unit metadata column to confirm price units
print("Price units:", corn_prices['unit_desc'].unique())

# NumPy functions work directly on a pandas Series column
print("\nMean nominal price: ", np.round(np.mean(corn_prices['nominal_price']), 2))
print("Std deviation:      ", np.round(np.std(corn_prices['nominal_price']), 2))
print("Min / Max:          ", np.min(corn_prices['nominal_price']),
      "/", np.max(corn_prices['nominal_price']))


In [ ]:
# .describe() generates count, mean, std, min, quartiles, and max for numeric columns
# .round(2) keeps output readable
corn_prices[['year', 'nominal_price']].describe().round(2)


## Practice

In [ ]:
#  Write a boolean filter that selects rows where:
#   - year is greater than 1970, AND
#   - nominal_price is greater than 3.0
# Then compute the mean nominal_price of that filtered subset.
#
# Hint: combine two conditions with & and wrap each in parentheses:
#   filtered = corn_prices[(condition1) & (condition2)]
#   print(filtered['nominal_price'].mean().round(2))


## Inflation Adjustment

To compare prices over time, we adjust nominal prices to real (inflation-adjusted) prices using the Consumer Price Index (CPI). We will use 2024 as the base year, so all prices are in terms of 2024 dollars.

The adjustment formula is: `real_price = (nominal_price / CPI_t) * CPI_2024`

### Merge CPI Data

In [ ]:
corn_prices = corn_prices.merge(
    cpi,  # CPI data loaded earlier
    left_on='year',  # Match on 'year' column in corn price data
    right_on='year',  # Match on 'Year' column in CPI data
    how='left'  # Keep all corn price records
)

### Drop Missing Values

Rows without a matching CPI year cannot be inflation-adjusted and are dropped.

In [ ]:
corn_prices.dropna(subset=['annual_avg_cpi'], inplace=True)

### Set the Base Year CPI

Retrieve the 2024 CPI value. This serves as the numerator in the inflation adjustment formula, scaling all prices up to 2024 purchasing power.

In [ ]:
cpi_base_year = cpi[cpi['year'] == 2024]['annual_avg_cpi'].values[0]  # Should be 313.689
print(cpi_base_year)

### Calculate Real Prices

Apply the CPI adjustment to produce `real_price` in 2024 dollars.

In [ ]:
corn_prices['real_price'] = (
    corn_prices['nominal_price'] / corn_prices['annual_avg_cpi']
) * cpi_base_year

In [ ]:
# Verify data frequency (should be annual)
print(corn_prices['freq_desc'].unique())

## Visualizations

The plots below show real corn prices over time. The first is a simple line plot; the second adds a regression line to highlight the long-run trend.

In [ ]:
# Set the style for seaborn plots
# 'whitegrid' provides a clean background with subtle gridlines
sns.set_style('whitegrid')

### Line Plot

Visualize real corn prices ($ / Bushel) over time

Documentation: [Seaborn Line Plot](https://seaborn.pydata.org/generated/seaborn.lineplot.html)

In [ ]:
plt.figure(figsize=(14, 6))

sns.lineplot(
    data=corn_prices,
    x='year',
    y='real_price',
    linewidth=1.0,
    color='darkblue'
)

plt.title("Annual Corn Prices ($/BU)", fontsize=14, weight='bold')
plt.xlabel("Year", fontsize=12, weight='bold')
plt.ylabel("Price ($/BU)", fontsize=12, weight='bold')
plt.xticks(rotation=90, ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# figsize=(16, 8) creates a wide canvas with room for both scatter points and the regression line
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(16, 8))

# regplot creates both scatterplot and fitted regression line
# ci=None turns off the confidence interval shading
sns.regplot(
    data=corn_prices,  # Use the merged dataframe
    x='year',          # Year on x-axis
    y='real_price',    # Real price on y-axis
    scatter_kws={
        'alpha': 0.7,          # Transparency: 0.7 makes points visible but not overpowering
        's': 60,               # Size of scatter points (60 is moderately sized)
        'color': 'darkblue',   # Color for price points
        'edgecolors': 'black', # Black border around points for definition
        'linewidths': 0.7,     # Border width
        'zorder': 3            # zorder=3 ensures points appear above lines
    },
    line_kws={
        'color': 'blue',       # Blue regression line
        'linewidth': 2.5,      # Thick line for visibility
        'linestyle': '--',     # Dashed line to distinguish from the connected series line
        'alpha': 0.7,          # Slight transparency
        'zorder': 2            # zorder=2 places regression line above grid but below points
    },
    ci=None,  # Turn off confidence interval shading
    ax=ax
)

# Add connected line to show year-to-year changes alongside the regression line
ax.plot(
    corn_prices['year'],
    corn_prices['real_price'],
    color='darkblue',
    linewidth=1.2,  # Thin so it doesn't overpower scatter and regression
    alpha=0.5,      # Semi-transparent to serve as background context
    zorder=1        # zorder=1 places connected line behind everything else
)

# fontsize=14, weight='bold', pad=15 keeps the title prominent and clear of the plot area
ax.set_title(
    'Corn Prices ($/BU)',
    fontsize=14,
    weight='bold',
    pad=15
)

ax.set_xlabel('Year', fontsize=12, weight='bold')
ax.set_ylabel('Price ($/BU)', fontsize=12, weight='bold')

# Show a tick every 3 years to avoid overlapping labels on the dense x-axis
all_years = sorted(corn_prices['year'].unique())
ax.set_xticks(all_years[::3])
ax.set_xticklabels(all_years[::3])

# labelsize=11 makes year and price values easier to read
# rotation=90 for x-axis prevents label overlap across the long date range
ax.tick_params(axis='y', labelsize=11)
ax.tick_params(axis='x', labelsize=11, rotation=90)

# Subtle gridlines (30% opacity, thin) aid value reading without dominating the plot
ax.grid(axis='both', alpha=0.3, linestyle='-', linewidth=0.5)

## Regression Modeling

Two fundamental questions we can ask of a price time series:

- **Linear time trend** (`real_price ~ t`): Does price drift up or down over time on average? This fits a straight line through the data using Ordinary Least Squares (OLS).
- **AR(1) - Autoregressive model**: Does this year's price depend on last year's? The AR(1) model says: `price_t = const + beta * price_{t-1} + error`. If beta is close to 1, prices are highly persistent.

We use `statsmodels` for both, which produces regression output with coefficients, standard errors, and p-values.

In [ ]:
# Use real (inflation-adjusted) prices for all modeling
# Sort ascending by year so that t=0 is the earliest observation (1913)
# This gives the OLS slope a clean interpretation: average $/bu change per year forward in time
df_model = corn_prices[['year', 'real_price']].dropna().sort_values('year').reset_index(drop=True).copy()
df_model['t'] = range(len(df_model))  # t=0 is 1913, t=1 is 1914, ...
df_model.head()


In [ ]:
import statsmodels.formula.api as smf

# The formula API automatically includes an intercept — no need to add it manually
# 't' is our integer time index: t=0 is 1913, t=1 is 1914, etc.
ols_result = smf.ols("real_price ~ t", data=df_model).fit()
print(ols_result.summary())


### Interpreting the OLS Output

Focus on the **`coef` column** in the results table:

- **`Intercept`**: the predicted price when `t = 0` (the first year in the dataset)
- **`t`**: the average annual change in real corn price — a negative (positive) coefficient means prices trend downward (upward) over time on average
- **`R-squared`**: the proportion of price variation explained by the time trend alone
- **`P>|t|`**: p-value — values below 0.05 suggest the coefficient is statistically significant


In [ ]:
from statsmodels.tsa.ar_model import AutoReg

# AR(1): this year's price is modeled as a function of last year's price
#   real_price_t = const + beta * real_price_{t-1} + error
# lags=1 means we use one lag; old_names=False uses current parameter naming
ar1_result = AutoReg(df_model['real_price'], lags=1, old_names=False).fit()
print(ar1_result.summary())


In [ ]:
# Recover the actual values used in fitting (fitted + residual)
y_actual = ar1_result.fittedvalues + ar1_result.resid

ss_res = np.sum(ar1_result.resid ** 2)          # sum of squared residuals
ss_tot = np.sum((y_actual - y_actual.mean()) ** 2)    # total sum of squares

r2_ar1 = 1 - ss_res / ss_tot
print(f"OLS R²:   {ols_result.rsquared:.4f}")
print(f"AR(1) R²: {r2_ar1:.4f}")

### Interpreting the AR(1) Output

Focus on the `real_price.L1` coefficient (the "L" stands for **lag**):

- A value near **1.0** means prices are highly persistent, implying last year's price is a very strong predictor of this year's. This is common in commodity markets.
- The `const` term is the baseline level when the lagged price equals zero (rarely meaningful by itself).
- Compare the AR(1) R² to the OLS R² — which model explains more variance in prices?


## Train/Test Split: Predicting the Last 5 Years

A **train/test split** is a fundamental evaluation method in predictive modeling: fit the model on one portion of the data and measure its accuracy on a held-out portion it has never seen.

For time series data, our split should always respect temporal order, because using future data to train a model that predicts the past will produce misleadingly optimistic results.

Here we hold out the last 5 years (2020-2024) as the test set and train on all prior years.

In [ ]:
# Hold out the last 5 years (2020–2024) as the test set
# Temporal order is preserved
cutoff_year = 2019
train = df_model[df_model['year'] <= cutoff_year].copy()
test  = df_model[df_model['year'] >  cutoff_year].copy()

print(f"Training set: {train['year'].min()}–{train['year'].max()} ({len(train)} observations)")
print(f"Test set:     {test['year'].min()}–{test['year'].max()}  ({len(test)} observations)")


In [ ]:
# Fit the linear trend model on training data only — test data is never seen during training
ols_train = smf.ols("real_price ~ t", data=train).fit()

# Predict on test rows: pass the test DataFrame with the same 't' column
# The formula API automatically finds the 't' values it needs
test['ols_predicted'] = ols_train.predict(test)
test[['year', 'real_price', 'ols_predicted']]


In [ ]:
# Fit AR(1) on training data only
ar1_train = AutoReg(train['real_price'], lags=1, old_names=False).fit()

# Forecast out-of-sample using integer positions in the series
# start=len(train) is the first observation after training ends
# dynamic=True: each forecast step uses the prior *predicted* value (true multi-step forecast)
ar1_preds = ar1_train.predict(
    start=len(train),
    end=len(train) + len(test) - 1,
    dynamic=True
)
test['ar1_predicted'] = ar1_preds.values
test[['year', 'real_price', 'ols_predicted', 'ar1_predicted']]


In [ ]:
# Root Mean Squared Error (RMSE): measures average prediction error in the same units as the data
# Lower RMSE = better forecast accuracy
rmse_ols = np.sqrt(np.mean((test['real_price'] - test['ols_predicted']) ** 2))
rmse_ar1 = np.sqrt(np.mean((test['real_price'] - test['ar1_predicted']) ** 2))

print(f"OLS trend RMSE: ${rmse_ols:.2f}/bu")
print(f"AR(1)     RMSE: ${rmse_ar1:.2f}/bu")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Training period: the data the models were fit on
ax.plot(train['year'], train['real_price'],
        color='steelblue', linewidth=1.2, label='Training data (1913–2019)')

# Test period: what actually happened
ax.plot(test['year'], test['real_price'],
        color='black', linewidth=2.5, marker='o', label='Actual (2020–2024)')

# OLS forecast: extrapolates the long-run linear trend
ax.plot(test['year'], test['ols_predicted'],
        color='crimson', linestyle='--', linewidth=2, marker='s',
        label='OLS trend forecast')

# AR(1) forecast: rolls forward from the last observed training value
ax.plot(test['year'], test['ar1_predicted'],
        color='darkorange', linestyle=':', linewidth=2, marker='^',
        label='AR(1) forecast')

# Vertical dashed line showing where training ends and test begins
ax.axvline(cutoff_year, color='gray', linestyle=':', linewidth=1.2,
           label=f'Train/test cutoff ({cutoff_year})')

ax.set_title(
    'Real Corn Prices: OLS Trend vs. AR(1) Forecast — Last 5 Years Holdout',
    fontsize=13, weight='bold'
)
ax.set_xlabel('Year', fontsize=11, weight='bold')
ax.set_ylabel('Real Price (2024 $/BU)', fontsize=11, weight='bold')
ax.legend()
plt.tight_layout()
plt.show()
